# Exploratory analyses

Two exploratory analyses. They suggest ideas; they do not confirm anything.

- **Number of parties held accountable (Q11):** does the number chosen (one, two or three) differ between groups? Chi-square test with Cramer's V and a 95% bootstrap interval.
- **Links between views:** Spearman correlations between the eight ordered answers, with 95% confidence intervals (Fisher z with the Bonett-Wright standard error) and Benjamini-Hochberg correction across the 28 pairs.

**Reads:** `data/survey_clean.csv`

**Writes to `results/`:** `accountability_number_test.csv`, `correlations.csv`, `correlation_matrix.csv`

In [1]:
from itertools import combinations
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

DATA_FILE = Path("../data/survey_clean.csv")
RESULTS = Path("../results")
RESULTS.mkdir(exist_ok=True)

SEED = 2026
N_BOOT = 2000
GROUPS = ["Patient", "Healthcare professional", "AI developer"]

# Ordered answers (codes from the cleaned data; Q7, Q9, Q14 and Q16 coded 1-5 in meaning order).
VIEWS = {
    "familiarity_code": "Familiarity with AI",
    "privacy_protection_code": "Privacy is protected (agree)",
    "trust_code": "Trust in AI",
    "autonomy_undermined_code": "Autonomy could be undermined (agree)",
    "bias_concern_code": "Bias concern",
    "explanation_importance_code": "Explanations important",
    "guidelines_confidence_code": "Guidelines are enough (confidence)",
    "disparities_concern_code": "Disparities concern",
}

df = pd.read_csv(DATA_FILE)
df["group"] = pd.Categorical(df["group"], categories=GROUPS)
assert df[list(VIEWS)].notna().all().all()
print("Participants:", len(df))

Participants: 253


## Number of parties held accountable, by group

In [2]:
rng = np.random.default_rng(SEED)


def cramers_v(t):
    chi2 = stats.chi2_contingency(t, correction=False)[0]
    return float(np.sqrt(chi2 / (t.sum() * (min(t.shape) - 1))))


table = pd.crosstab(df["acc_n_selected"], df["group"]).reindex(columns=GROUPS, fill_value=0)
expected = stats.contingency.expected_freq(table.values)
if (expected < 5).mean() <= 0.20 and (expected >= 1).all():
    res = stats.chi2_contingency(table.values, correction=False)
    test, stat, dof, p = "Chi-square", float(res.statistic), int(res.dof), float(res.pvalue)
else:
    res = stats.fisher_exact(table.values, method=stats.MonteCarloMethod(n_resamples=10000, rng=rng))
    test, stat, dof, p = "Fisher exact", np.nan, np.nan, float(res.pvalue)

parts = [df[df["group"] == g] for g in GROUPS]
boot = []
for _ in range(N_BOOT):
    s = pd.concat([x.iloc[rng.integers(0, len(x), len(x))] for x in parts])
    boot.append(cramers_v(pd.crosstab(s["acc_n_selected"], s["group"]).reindex(index=[1, 2, 3], columns=GROUPS, fill_value=0).values))
v_lo, v_hi = np.percentile(boot, [2.5, 97.5])

result = pd.DataFrame([{"test": test, "statistic": stat, "df": dof, "p": p, "cramers_v": cramers_v(table.values),
                        "ci_low": v_lo, "ci_high": v_hi, "min_expected": expected.min()}])
result.to_csv(RESULTS / "accountability_number_test.csv", index=False)
print("Counts (rows: number of parties chosen):")
print(table.to_string())
result.round(3)

Counts (rows: number of parties chosen):
group           Patient  Healthcare professional  AI developer
acc_n_selected                                                
1                    21                       33            57
2                    10                       14            42
3                    17                       11            48


,test,statistic,df,p,cramers_v,ci_low,ci_high,min_expected
0,Chi-square,7.217,4,0.125,0.119,0.067,0.22,12.522


## Links between views (Spearman correlations)

In [3]:
n = len(df)
rows = []
for a, b in combinations(VIEWS, 2):
    rho, p = stats.spearmanr(df[a], df[b])
    se = np.sqrt((1 + rho ** 2 / 2) / (n - 3))          # Bonett-Wright standard error
    z = np.arctanh(rho)
    rows.append({"var_1": VIEWS[a], "var_2": VIEWS[b], "rho": rho,
                 "ci_low": np.tanh(z - 1.96 * se), "ci_high": np.tanh(z + 1.96 * se), "p": p})
corr = pd.DataFrame(rows)
corr["p_bh"] = multipletests(corr["p"], method="fdr_bh")[1]
corr["after_bh"] = corr["p_bh"] < 0.05
corr = corr.sort_values("rho", key=abs, ascending=False).reset_index(drop=True)
corr.to_csv(RESULTS / "correlations.csv", index=False)

matrix = df[list(VIEWS)].corr(method="spearman").rename(index=VIEWS, columns=VIEWS).round(3)
matrix.to_csv(RESULTS / "correlation_matrix.csv")

print("Pairs with p < 0.05 after Benjamini-Hochberg:", int(corr["after_bh"].sum()), "of", len(corr))
corr.round(3)

Pairs with p < 0.05 after Benjamini-Hochberg: 16 of 28


,var_1,var_2,rho,ci_low,ci_high,p,p_bh,after_bh
0,Privacy is protected (agree),Trust in AI,0.404,0.291,0.506,0.000,0.000,True
1,Familiarity with AI,Explanations important,0.300,0.181,0.411,0.000,0.000,True
2,Bias concern,Disparities concern,0.278,0.158,0.390,0.000,0.000,True
3,Autonomy could be undermined (agree),Bias concern,0.256,0.135,0.369,0.000,0.000,True
4,Privacy is protected (agree),Guidelines are enough (confidence),0.237,0.116,0.352,0.000,0.001,True
5,Trust in AI,Explanations important,0.230,0.108,0.345,0.000,0.001,True
6,Trust in AI,Guidelines are enough (confidence),0.227,0.106,0.343,0.000,0.001,True
7,Explanations important,Disparities concern,0.204,0.082,0.321,0.001,0.004,True
8,Bias concern,Explanations important,0.192,0.069,0.309,0.002,0.007,True
9,Autonomy could be undermined (agree),Explanations important,0.188,0.065,0.305,0.003,0.007,True
